# 01 Data: Download of stock prices and macro data

**Project 1:** Predicting stock returns using machine learning

Dieses Notebook lädt alle Rohdaten, prüft ihre Qualität und aggregiert sie auf Monatsebene.
Die Rohdaten werden in `data/raw/` zwischengespeichert, damit nicht bei jedem Lauf neu heruntergeladen wird.

| Step | Output |
|------|--------|
| Daily stock prices (Yahoo Finance) | `data/raw/prices_daily.csv` |
| Market indices S&P 500 and VIX (Yahoo Finance) | `data/raw/market_daily.csv` |
| Macro series (FRED) | `data/raw/macro_daily.csv` |
| Monthly aggregation | `data/processed/*_monthly.csv` |

## Libraries and settings

In [ ]:
# Libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Make the project folder importable (notebook lives in /notebooks)
sys.path.append(os.path.abspath('..'))
from src import config, data

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Settings
REFRESH = False  # True = download again even if a file already exists
RAW_DIR = '../data/raw'
PROC_DIR = '../data/processed'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

# Show current working directory
print(os.getcwd())

## Settings of the universe
Die Aktienauswahl und der Zeitraum stehen zentral in `src/config.py`.

In [ ]:
print('Number of stocks:', len(config.TICKERS))
print('Start:', config.START, '| End:', config.END or 'today')
print('First 10 tickers:', config.TICKERS[:10])

## Download daily stock prices (Yahoo Finance)
`close` ist bereits um Splits und Dividenden bereinigt (`auto_adjust=True`). Das ist wichtig, damit Renditen korrekt sind.

In [ ]:
path_prices = f'{RAW_DIR}/prices_daily.csv'

if os.path.exists(path_prices) and not REFRESH:
    prices = pd.read_csv(path_prices, parse_dates=['date'])
    print('Loaded from file:', path_prices)
else:
    prices = data.download_prices(config.TICKERS, config.START, config.END)
    prices.to_csv(path_prices, index=False)
    print('Downloaded and saved:', path_prices)

print(prices.shape)
prices.head()

## Download market indices (S&P 500, VIX)

In [ ]:
path_market = f'{RAW_DIR}/market_daily.csv'

if os.path.exists(path_market) and not REFRESH:
    market = pd.read_csv(path_market, parse_dates=['date'])
    print('Loaded from file:', path_market)
else:
    market = data.download_prices(config.MARKET_TICKERS, config.START, config.END)
    market.to_csv(path_market, index=False)
    print('Downloaded and saved:', path_market)

print(market.shape)
market.groupby('ticker')['date'].agg(['min', 'max', 'count'])

## Download macro data (FRED)

In [ ]:
path_macro = f'{RAW_DIR}/macro_daily.csv'

if os.path.exists(path_macro) and not REFRESH:
    macro = pd.read_csv(path_macro, index_col=0, parse_dates=True)
    print('Loaded from file:', path_macro)
else:
    macro = data.download_fred(list(config.FRED_SERIES.keys()), config.START)
    macro.to_csv(path_macro)
    print('Downloaded and saved:', path_macro)

print(macro.shape)
macro.tail()

## Data quality checks
Wir prüfen pro Aktie: Länge der Historie, fehlende Werte und extreme Tagesrenditen (mögliche Datenfehler).
Aktien mit kurzer Historie sind hier oben sortiert.

In [ ]:
report = data.quality_report(prices)
report.head(10)

In [ ]:
# Summary of potential problems
print('Stocks with less than 3000 trading days:', (report['n_days'] < 3000).sum())
print('Missing close values in total:           ', int(report['n_missing_close'].sum()))
print('Stocks with a daily move of more than 25%:', (report['max_abs_daily_ret'] > 0.25).sum())

# Which stocks have extreme daily moves?
report[report['max_abs_daily_ret'] > 0.25][['max_abs_daily_ret']].round(3)

**Interpretation:** Extreme Tagesbewegungen sind bei einzelnen Aktien echte Ereignisse (z.B. Krisentage im März 2020 oder Earnings-Überraschungen).
Wir entfernen sie hier **nicht**, sondern behandeln Ausreißer später im Feature-Schritt (z.B. durch Winsorizing der Renditen).

## Aggregate to monthly data
Das Modell sagt die Rendite des **Folgemonats** voraus. Dafür brauchen wir Monatsendwerte.

In [ ]:
prices_monthly = data.to_monthly(prices)
market_monthly = data.to_monthly(market)
macro_monthly = data.macro_to_monthly(macro)

print('Stocks, monthly:', prices_monthly.shape)
print('Market, monthly:', market_monthly.shape)
print('Macro, monthly: ', macro_monthly.shape)
prices_monthly.head()

In [ ]:
# Save processed data
prices_monthly.to_csv(f'{PROC_DIR}/prices_monthly.csv', index=False)
market_monthly.to_csv(f'{PROC_DIR}/market_monthly.csv', index=False)
macro_monthly.to_csv(f'{PROC_DIR}/macro_monthly.csv')
print('Saved to', PROC_DIR)

## Quick look at the data

In [ ]:
# Number of stocks with data per month (universe grows when stocks start later)
n_per_month = prices_monthly.groupby('date')['ticker'].nunique()

plt.figure(figsize=(7, 3))
plt.plot(n_per_month.index, n_per_month.values, color='darkred')
plt.title('Number of stocks with data per month')
plt.xlabel('month')
plt.ylabel('number of stocks')
plt.grid()
plt.show()

In [ ]:
# Distribution of monthly returns of all stocks
rets = prices_monthly['ret_1m'].dropna()
print(rets.describe().round(4))

plt.figure(figsize=(7, 3))
plt.hist(rets.clip(-0.5, 0.5), bins=80, color='blue', alpha=0.5)
plt.title('Histogram of monthly returns (clipped at -50% / +50%)')
plt.xlabel('monthly return')
plt.ylabel('frequency')
plt.grid()
plt.show()

In [ ]:
# Macro series over time
macro_monthly.plot(subplots=True, figsize=(8, 8), grid=True, legend=True)
plt.tight_layout()
plt.show()

## Summary
- Rohdaten (täglich) liegen in `data/raw/`, Monatsdaten in `data/processed/`.
- Die Rendite `ret_1m` ist die einfache Monatsrendite. Das Ziel (Rendite des *nächsten* Monats) wird erst in `02_features` gebildet, damit keine Zukunftsinformation in die Features gelangt.
- Bekannte Limitationen (für den Report): Survivorship Bias durch die Auswahl heutiger Large Caps, keine historischen Fundamentaldaten bei Yahoo, Makrodaten werden mit Verzögerung veröffentlicht.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')